# scProto + cell-cell similarity reconstruction (`lambda_sim_recon`) — train & eval

Self-contained notebook for one experiment: does adding the cell-cell
similarity reconstruction loss close the resolution gap between scProto and
SEACells? Background: SEACells' archetypal analysis reconstructs each cell's
full row of the affinity matrix (forcing archetypes to resolve genuine
per-cell heterogeneity); scProto's existing losses (nassoc, proto_usage)
only ever see a K×K summary, so a large, internally well-connected but
heterogeneous prototype pays no penalty. `lambda_sim_recon` (off by default,
see `configs/defaults.py`) adds that missing cell-level pressure: it decodes
each prototype into a predicted target and reconstructs it through `S`, with
both `S` and the prototypes kept trainable through this loss specifically
(see `scproto.py`'s `sim_recon_loss` block for the full reasoning on why
that differs from `proto_recon_loss`'s detached `S`).

Two target modes, both trained and compared here (`sim_recon_target`):
- **`full`** — reconstruct each cell's actual row of the affinity graph.
  Most literal match to what SEACells' RSS objective does.
- **`diffusion`** — regress to a small precomputed per-cell diffusion-map
  coordinate instead of the full row. Cheaper (no O(n_cells) decoder
  output), but the compression could in principle discard some of the
  resolving signal the `full` target has — that's an open question, not
  an assumption, which is why both get trained and compared side by side
  rather than picking one up front.

**Scope for this pass: `arbf` only.** This trains exactly 3 scProto configs
(arbf baseline, +full, +diffusion) plus SEACells — one affinity at a time
keeps each pass cheap and isolates whether either sim-recon target closes
the gap with SEACells before spending Colab time on other affinities.

Runs are matched for evaluation by **exact** saved model directory name
(captured right after training/loading), not by keyword substring — a
baseline run's name is always a literal substring of both its sim-recon
counterparts' names, so fuzzy keyword matching (as used in
`scproto_spatial_comparison.ipynb`, safe there because it only ever had one
variant per affinity) would silently match the wrong run. Exact-name
matching sidesteps that entirely.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q scarches SEACells faiss-gpu-cu12 scib-metrics

In [3]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


## Config

In [4]:
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.spatial_immune_task import NSCLC_EVAL_GROUPS
import pandas as pd

DS_ID = 's28nsc'
AFFINITY = 'arbf'

# lambda_sim_recon=1.0 is a first guess — affinity values already live in
# ~[0,1], comparable scale to nassoc's own terms. Tune from here if purity
# doesn't move, or moves too aggressively at the cost of the other losses.
LAMBDA_SIM_RECON = 1.0
SIM_RECON_N_EIGS = 128  # only used by sim_recon_target='diffusion'

COMMON_KWARGS = dict(
    cvae_epochs=50,
    train_epochs=50,
    eval_freq=3,
    patience=6,
    batch_size=1024,
    umap_steps_per_epoch=500,
    niche_key='niches_2D',
    target_groups=NSCLC_EVAL_GROUPS,
    lambda_config=LAMBDA_PROTO_UMAP_PRECON | {'nassoc_agg': 'max'},
)

# Baseline: set True if you already trained arbf in
# train_scproto_spatial.ipynb and just want to reload that checkpoint here.
# Set False to train it fresh in this notebook (fully self-contained, just slower).
LOAD_BASELINE = True

# The sim-recon runs (both targets) are the new thing this notebook exists to
# produce — False trains them; flip to True on a re-run to just reload.
LOAD_SIMRECON = False

trainers = {}
results = {}
mc_adatas = {}
model_names = {}  # label -> exact saved model directory name (see note above on why exact, not keyword)


def train_or_load(label, affinity_type, load, extra_lambda=None):
    """Run (or reload) one scProto config and record its exact model directory name."""
    kwargs = COMMON_KWARGS if not extra_lambda else COMMON_KWARGS | {
        'lambda_config': COMMON_KWARGS['lambda_config'] | extra_lambda
    }
    t, res, mc_ad = run_mc_task(DS_ID, affinity_type=affinity_type, load_umap=load, **kwargs)
    trainers[label], results[label], mc_adatas[label] = t, res, mc_ad
    model_names[label] = t.get_model_name()
    print(f'{label}: {res}')
    return t, res, mc_ad

## Train / load — arbf, three configs

Each cell is independent — skip/re-run any one without affecting the others.

### Baseline (no sim-recon)

In [5]:
train_or_load('arbf', AFFINITY, load=LOAD_BASELINE)

 captum (see https://github.com/pytorch/captum).


dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

📊 Affinity: wdeg[min/mean/max]=1.315/25.499/124.874, effk_med=61.6, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  n_eigs=10  nnz=4352008  nnz/row=74.5  w[min/mean/max]=3.114e-03/3.423e-01/9.835e-01  deg[min/mean/max]=1.31/25.50/124.87
[w

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 969.27proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

...

Saved metacells (800 prototypes) to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
arbf: {'purity': 0.8619394497269813, 'niche_purity': 0.8559889901645817, 'batch_entropy': -1.0000000826903712e-10, 'modularity': 0.46529812761429373, 'coverage': 1.0, 'dge_rbo_avg': 0.17448630265777365, 'dge_kendall_avg': 0.05627731316780108, 'dge_jaccard_avg': 0.09334003857560122, 'scgraph_corr_avg': 0.3536934921200777, 'ct_niche_rbo_avg': 0.02809004239715028, 'aff_compactness_per_batch': {'section_28': 0.2120796053693693}, 'aff_compactness_mean': 0.13513359875958822, 'tumor_cells_tumor_surface_purity': 0.34455557527945607, 'tumor_cells_tumor_surface_coverage': 0.5686202461877641, 'tumor_cells_tumor_surface_homogeneity': 0.5632923020393166, 'tumor_cells_tumor_core_purity': 0.33486030630110225, 'tumor_cells_tumor_core_coverage': 0.3832035595105673, 'tumor_cells_tumor_core_homogeneity': 0.6031701890989989, 'm

(<interpretable_ssl.trainers.scproto.SCProtoTrainer at 0x7af33b6736b0>,
 {'purity': 0.8619394497269813,
  'niche_purity': 0.8559889901645817,
  'batch_entropy': -1.0000000826903712e-10,
  'modularity': 0.46529812761429373,
  'coverage': 1.0,
  'dge_rbo_avg': 0.17448630265777365,
  'dge_kendall_avg': 0.05627731316780108,
  'dge_jaccard_avg': 0.09334003857560122,
  'scgraph_corr_avg': 0.3536934921200777,
  'ct_niche_rbo_avg': 0.02809004239715028,
  'aff_compactness_per_batch': {'section_28': 0.2120796053693693},
  'aff_compactness_mean': 0.13513359875958822,
  'tumor_cells_tumor_surface_purity': 0.34455557527945607,
  'tumor_cells_tumor_surface_coverage': 0.5686202461877641,
  'tumor_cells_tumor_surface_homogeneity': 0.5632923020393166,
  'tumor_cells_tumor_core_purity': 0.33486030630110225,
  'tumor_cells_tumor_core_coverage': 0.3832035595105673,
  'tumor_cells_tumor_core_homogeneity': 0.6031701890989989,
  'macrophages_macrophage_islands_purity': 0.24645234131327054,
  'macrophages_mac

### +sim-recon (`full` target)

Everything else identical to the baseline above — a clean single-variable
ablation. Reconstructs each cell's actual affinity-graph row.

In [6]:
train_or_load('arbf+full', AFFINITY, load=LOAD_SIMRECON,
               extra_lambda={'lambda_sim_recon': LAMBDA_SIM_RECON, 'sim_recon_target': 'full'})

dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_lsr1.0_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

📊 Affinity: wdeg[min/mean/max]=1.315/25.499/124.874, effk_med=61.6, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  n_eigs=10  nnz=4352008  nnz/row=74.5  w[min/mean/max]=3.114e-03/3.423e-01/9.835e-01  deg[min/mean/max]=1.31/25.50/12

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 949.83proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

...

Saved metacells (800 prototypes) to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_lsr1.0_upm-dotp_v31/metacells.h5ad


  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//s28nsc/proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_lsr1.0_upm-dotp_v31
arbf+full: {'purity': 0.8993952582602305, 'niche_purity': 0.8750253102903839, 'batch_entropy': -1.0000000826903713e-10, 'modularity': 0.4538989275080423, 'coverage': 0.9444444444444444, 'dge_rbo_avg': 0.07985248436823472, 'dge_kendall_avg': 0.08265718798718052, 'dge_jaccard_avg': 0.12447472358836162, 'scgraph_corr_avg': 0.36846765939881315, 'ct_niche_rbo_avg': 0.019942009270803997, 'aff_compactness_per_batch': {'section_28': 0.216131945447816}, 'aff_compactness_mean': 0.10661082857514437, 'tumor_cells_tumor_surface_purity': 0.32541128724867635, 'tumor_cells_tumor_surface_coverage': 0.5934227448098475, 'tumor_cells_tumor_surface_homogeneity': 0.5882785228734154, 'tumor_cells_tumor_core_purity': 0.354532783405382, 'tumor_cells_tumor_core_coverage': 0.0033370411568409346, 'tumor_cells_tumor_core_homogene

(<interpretable_ssl.trainers.scproto.SCProtoTrainer at 0x7af2d0db3710>,
 {'purity': 0.8993952582602305,
  'niche_purity': 0.8750253102903839,
  'batch_entropy': -1.0000000826903713e-10,
  'modularity': 0.4538989275080423,
  'coverage': 0.9444444444444444,
  'dge_rbo_avg': 0.07985248436823472,
  'dge_kendall_avg': 0.08265718798718052,
  'dge_jaccard_avg': 0.12447472358836162,
  'scgraph_corr_avg': 0.36846765939881315,
  'ct_niche_rbo_avg': 0.019942009270803997,
  'aff_compactness_per_batch': {'section_28': 0.216131945447816},
  'aff_compactness_mean': 0.10661082857514437,
  'tumor_cells_tumor_surface_purity': 0.32541128724867635,
  'tumor_cells_tumor_surface_coverage': 0.5934227448098475,
  'tumor_cells_tumor_surface_homogeneity': 0.5882785228734154,
  'tumor_cells_tumor_core_purity': 0.354532783405382,
  'tumor_cells_tumor_core_coverage': 0.0033370411568409346,
  'tumor_cells_tumor_core_homogeneity': 0.8378754171301446,
  'macrophages_macrophage_islands_purity': 0.2197638915661853,
  '

### +sim-recon (`diffusion` target)

Same ablation, but regresses to a precomputed `SIM_RECON_N_EIGS`-dim
diffusion-map coordinate per cell instead of the full affinity row —
cheaper, and the question this notebook exists to answer is whether that
compression costs any of the resolution benefit the `full` target gives.

In [7]:
SIM_RECON_N_EIGS

128

In [ ]:
train_or_load('arbf+diffusion', AFFINITY, load=LOAD_SIMRECON,
               extra_lambda={'lambda_sim_recon': LAMBDA_SIM_RECON, 'sim_recon_target': 'diffusion',
                              'sim_recon_n_eigs': 512})

dataset is None, loading s28nsc
loading s28nsc data
⚠️ No HVG column found.
proto_umap_ds-s28n_NP800_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_lsr1.0_srt-diff_srne512_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 

📊 Affinity: wdeg[min/mean/max]=1.315/25.499/124.874, effk_med=61.6, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/s28nsc/pretrain/pretrain_ds-s28nsc_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 's28nsc', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'section'}
[waypoint init] N=58423  K=800  n_eigs=10  nnz=4352008  nnz/row=74.5  w[min/mean/max]=3.114e-03/3.423e-01/9.835e-01  deg[min/mean/m

waypoint MaxMin: 100%|██████████| 799/799 [00:00<00:00, 947.86proto/s]


[waypoint init] selected 800 seed cells


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

[eps calibration] E[p_pos]=0.3427 unreachable (max E[q_pos]=0.0162), falling back to effk=5.0


  0%|          | 0/58 [00:00<?, ?it/s]

[effk calibration] target_effk=5.0 → epsilon=0.0299 (mean_effk=5.00)
📊 EdgeDataset: 4351982 edges
   Weight range: [0.0051, 0.9835]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 4351982 total)
📐 UMAP kernel: min_dist=0.5, spread=1.0 -> a=0.5830, b=1.3342


## SEACells (PCA) baseline

`train_seacell` skips training and returns immediately if a saved run is
already found — safe to call every time.

In [ ]:
train_seacell(DS_ID, mode='eval', build_kernel_on='X_pca')

## Quick numeric comparison — the three in-memory scProto runs

Straight from the metrics `run_mc_task` already returned — no disk lookup,
so this part can't be affected by the exact-name-matching note above.

In [ ]:
metrics_df = pd.DataFrame(results).T
metrics_df

## Full comparison — cell-type purity vs. niche purity, incl. SEACells

Same metric definitions and plots as `scproto_spatial_comparison.ipynb`
(see that notebook's markdown for the exact `celltype_purity` /
`niche_purity` formulas) — reproduced here so this notebook is a complete,
standalone record of the experiment. `NICHE_KEY='niches_2D'` here by choice;
it doesn't need to match training's own `niche_key` (`niches_3D` above) —
this just scores purity against whichever ground-truth niche column you
point it at.

In [ ]:
NICHE_KEY = 'niches_2D'
CELLTYPE_KEY = 'celltypes'
MIN_CELLS = 20

GRAPH_DIR = os.path.join(os.environ['CODE_DIR'], 'graphs')
os.makedirs(GRAPH_DIR, exist_ok=True)

# Exact model directory names, captured right after training/loading above —
# see the top-of-notebook note on why this must be exact, not a keyword.
MODEL_KEYWORDS = {
    model_names['arbf']:           'scProto (arbf)',
    model_names['arbf+full']:      'scProto + sim-recon/full (arbf)',
    model_names['arbf+diffusion']: 'scProto + sim-recon/diffusion (arbf)',
    'seacell':                     'SEACells (PCA)',
}
MODEL_KEYWORDS

### 1. Cell-type purity table

In [ ]:
median_ct, q25_ct, q75_ct = fig_celltype_purity_table(
    DS_ID, MODEL_KEYWORDS,
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS, metric='celltype_purity',
)
format_purity_table(median_ct, q25_ct, q75_ct)

In [ ]:
median_ct.round(3).style.background_gradient(cmap='YlGnBu', vmin=0, vmax=1)

### 2. Niche purity heatmap — one panel per model

In [ ]:
fig_all_celltype_niche_heatmap(
    DS_ID, MODEL_KEYWORDS,
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS, metric='niche_purity',
    save_path=os.path.join(GRAPH_DIR, 'sim_recon_niche_heatmap.pdf'),
)

### 3. Difference heatmap — does sim-recon beat the plain-arbf baseline?

(model − arbf-baseline) per (cell type, niche) cell — same convention as
`scproto_spatial_comparison.ipynb`. Two things to read off this one:
- Are `+full`/`+diffusion` redder than the baseline — the actual
  single-variable ablation this notebook exists to answer.
- Is `+full` redder than `+diffusion` — the direct answer to whether the
  diffusion compression costs resolution.

In [ ]:
fig_celltype_niche_heatmap_diff(
    DS_ID, MODEL_KEYWORDS, reference=model_names['arbf'],
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS, metric='niche_purity', min_n=5,
    save_path=os.path.join(GRAPH_DIR, 'sim_recon_niche_diff_heatmap.pdf'),
)

### 4. Trade-off — cell-type purity vs. niche purity, per model

In [ ]:
fig_celltype_niche_tradeoff(
    DS_ID, MODEL_KEYWORDS,
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS,
    save_path=os.path.join(GRAPH_DIR, 'sim_recon_tradeoff.pdf'),
)

### 5. Niche purity summary table

In [ ]:
median_niche, q25_niche, q75_niche = fig_celltype_purity_table(
    DS_ID, MODEL_KEYWORDS,
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS, metric='niche_purity',
)
format_purity_table(median_niche, q25_niche, q75_niche)

In [ ]:
median_niche.round(3).style.background_gradient(cmap='YlOrRd', vmin=0, vmax=1)

### 6. Distribution per cell type (violin)

In [ ]:
fig_purity_violin(
    DS_ID, MODEL_KEYWORDS,
    celltype_key=CELLTYPE_KEY, niche_key=NICHE_KEY,
    min_cells=MIN_CELLS, metric='niche_purity',
    save_path=os.path.join(GRAPH_DIR, 'sim_recon_niche_purity_violin.pdf'),
)

## Visualize — UMAP per run

Colored by cell type and (3D) niche, prototypes overlaid.

In [ ]:
for name, t in trainers.items():
    print(f'--- {name} ---')
    fig, proto_labels = t.plot_umap_simple(
        color_key=['celltypes', 'niches_3D'],
        show_proto_nums=False,
    )